# 문제 3-1 : 콘텐츠분쟁해결 RAG 시스템

`콘텐츠분쟁해결_사례.pdf`(한국소비자원·전자거래분쟁조정위원회 등의 **게임 / 이러닝 / 웹콘텐츠** 분쟁조정 사례집)를 근거로,
사용자의 분쟁 상황에 대해 **관련 사례·법령·해결 방안**을 답변하는 RAG 시스템입니다.

```
PDF ─ PyPDFLoader ─→ 페이지(목차 제외) ─ TextSplitter ─→ 청크 ─ Embeddings ─→ FAISS ─ Retriever(MMR)
                                                                                     │
질문 ───────────────────────── 검색(1회) → 프롬프트 → LLM → 답변 + 참조 사례(출처) ◀──┘
```

### 1) 라이브러리 설치 & API Key 로드

In [1]:
# uv add pypdf langchain-community langchain-text-splitters faiss-cpu langchain-openai langchain-upstage python-dotenv

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

# 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY), ("UPSTAGE_API_KEY", UPSTAGE_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)
UPSTAGE_API_KEY: 설정됨 (up...)


### 2) 설정값
- **임베딩 제공자**(`EMBEDDING_PROVIDER`)와 **LLM 제공자**(`PROVIDER`)는 가지고 있는 API Key 에 맞게 바꾸세요.
- 임베딩 모델을 바꾸면 기존 인덱스와 호환되지 않으므로 `DB_PATH` 가 제공자별로 분리되어 있습니다.

In [3]:
# ── 데이터 / 인덱스 경로 (연습문제 폴더 기준 상대 경로, 구분자는 슬래시 사용) ──
PDF_PATH = "../data/콘텐츠분쟁해결_사례.pdf"

# ── 임베딩 제공자: "openai" 또는 "upstage" ──
EMBEDDING_PROVIDER = "openai"
EMBEDDING_MODELS = {
    "openai": "text-embedding-3-large",
    "upstage": "solar-embedding-1-large",
}
DB_PATH = f"../db/faiss_content_dispute_{EMBEDDING_PROVIDER}"

# ── 분할 / 검색 설정 ──
CHUNK_SIZE = 1500      # 사례 1건(사건개요~처리결과)이 최대한 한 청크에 들어가도록 크게 설정
CHUNK_OVERLAP = 300    # 청크 경계에서 맥락이 끊기지 않도록 중복
TOP_K = 5              # LLM 에 전달할 청크 수
FETCH_K = 20           # MMR 이 후보로 가져올 청크 수 (이 중에서 다양성을 고려해 TOP_K 개 선택)

print(PDF_PATH, DB_PATH, EMBEDDING_MODELS[EMBEDDING_PROVIDER], sep="\n")

../data/콘텐츠분쟁해결_사례.pdf
../db/faiss_content_dispute_openai
text-embedding-3-large


### 3) LLM 생성 함수 (수업 예제 공통 설정)

In [4]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage"
PROVIDER = "openai"

LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o",  # 비용을 줄이려면 "gpt-4o-mini"
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5


def get_llm(temperature: float = 0.2, provider: str | None = None, verbose: bool = True):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = cls(**config, temperature=temperature, max_retries=MAX_RETRIES)
    if verbose:
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

### 4) 문서 로딩 (PyPDFLoader) — 빈 페이지·목차 페이지 제외
PDF 앞부분의 표지(빈 페이지)와 목차(`······` 점선) 페이지는 검색 상위에 끼어들어 자리를 낭비하므로 제외하고,
모든 페이지에 반복되는 머리말(`콘텐츠분쟁조정 법리 연구 2부 ...`)도 제거합니다.

In [5]:
import re
from langchain_community.document_loaders import PyPDFLoader

print("==> 1. 문서 로딩 → PDF 읽기...")
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"  총 {len(pages)}페이지 로드 완료")


def is_useful_page(doc) -> bool:
    """빈 페이지(텍스트 50자 미만)와 목차 페이지(점선 '····' 이 많은 페이지)를 걸러냄"""
    text = doc.page_content.strip()
    return len(text) >= 50 and text.count("····") < 5


# 모든 페이지 위쪽에 반복되는 머리말(문서 제목 + 쪽 번호)은 검색에 방해가 되므로 제거
HEADER_PATTERN = re.compile(r"^\s*콘텐츠분쟁조정 법리 연구\s*2부\s*-\s*타 분쟁조정사례 조사\s*-\s*\n?\s*\d*\s*\n?")

documents = []
for doc in pages:
    if is_useful_page(doc):
        doc.page_content = HEADER_PATTERN.sub("", doc.page_content).strip()
        documents.append(doc)
print(f"  빈 페이지/목차 제외 후 {len(documents)}페이지 사용 (제외 {len(pages) - len(documents)}페이지)")
print("\n[예시] P", documents[5].metadata.get("page_label"), documents[5].page_content[:300])

C:\Users\user\AppData\Local\Temp\ipykernel_21920\3387885815.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


==> 1. 문서 로딩 → PDF 읽기...
  총 109페이지 로드 완료
  빈 페이지/목차 제외 후 95페이지 사용 (제외 14페이지)

[예시] P 14 인한 피해가 미약하여 신청인의 주장을 받아들이기 어려우나 , 평소 접속하는 IP와 완전히 
다른 IP에서 접속했을 당시 불법프로그램이 사용된 점을 감안하여 계정 블록은 해지하고 불
법프로그램을 사용하여 이득을 본 골드, 경험치는 삭체처리 하겠다고 하였다 .
【처리결과 】
신청인이 상기사항을 수용하기로 함.


### 5) 문서 분할 (RecursiveCharacterTextSplitter)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("==> 2. 문서 분할 → 작은 청크로 나누기")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],  # 자연스러운 분할을 위한 구분자
)
chunks = text_splitter.split_documents(documents)

print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f}자")

==> 2. 문서 분할 → 작은 청크로 나누기
  95개 청크 생성 완료
  평균 청크 길이: 749자


### 6) 임베딩 + FAISS 벡터스토어 (인덱스가 있으면 로드, 없으면 생성)
처음 실행할 때만 임베딩 API 가 호출되고, 두 번째부터는 저장된 인덱스를 불러옵니다.

In [7]:
from pathlib import Path
from langchain_community.vectorstores import FAISS


def get_embeddings(provider: str = EMBEDDING_PROVIDER):
    """임베딩 제공자에 맞는 임베딩 모델 생성"""
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=EMBEDDING_MODELS["openai"])
    if provider == "upstage":
        from langchain_upstage import UpstageEmbeddings
        return UpstageEmbeddings(model=EMBEDDING_MODELS["upstage"])
    raise ValueError(f"지원하지 않는 임베딩 제공자: {provider}")


print("==> 3. 벡터화 → 임베딩으로 변환")
embeddings = get_embeddings()

print("==> 4. 저장 → FAISS 벡터스토어")
if Path(DB_PATH).exists():
    # 직접 만든 인덱스이므로 역직렬화 허용
    vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    print(f"  기존 인덱스 로드: {DB_PATH} ({vectorstore.index.ntotal}개 벡터)")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(DB_PATH)
    print(f"  인덱스 생성 및 저장: {DB_PATH} ({len(chunks)}개 벡터)")

==> 3. 벡터화 → 임베딩으로 변환
==> 4. 저장 → FAISS 벡터스토어
  기존 인덱스 로드: ../db/faiss_content_dispute_openai (95개 벡터)


### 7) 검색기 (Retriever) — MMR 검색
비슷한 사례가 많은 문서라 단순 유사도 검색은 같은 사례의 청크만 여러 개 가져오기 쉽습니다.
**MMR**(Maximal Marginal Relevance)은 관련성과 다양성을 함께 고려해 서로 다른 사례를 고르게 가져옵니다.

In [8]:
print("==> 5. 검색 → 질문과 유사한 문서 찾기")
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": FETCH_K, "lambda_mult": 0.7},  # lambda_mult: 1에 가까울수록 관련성 중시
)

# 검색 테스트
for doc in retriever.invoke("게임 아이템이 해킹으로 사라졌어요"):
    print(f"  P{doc.metadata.get('page_label', '?')}: {doc.page_content[:70].replace(chr(10), ' ')}...")

==> 5. 검색 → 질문과 유사한 문서 찾기
  P31: 1. 게임 25■ 1-가-1) 한국소비자원 조정전 상담사례 2009_해킹으로 손실된 아이템 복구 요청 【사건개요 】 피청구인이...
  P22: ■ 1-가-1) 한국소비자원 조정전 상담사례 2008_해킹으로 소실된 아이템 복원 요구 【사건개요 】 신청인은 피신청인의 인터...
  P26: ■ 1-가-1) 한국소비자원 조정전 상담사례 2009_사용하지 않았으나 차감된 아이템 배상 요구 【사건개요 】 신청인은 온라인...
  P11: 1. 게임 5■ 1-가-1) 한국소비자원 조정전 상담사례 2007_인터넷게임서비스 아이템 복구 요구 【사건개요 】 신청인은 피...
  P19: 1. 게임 13■ 1-가-1) 한국소비자원 조정전 상담사례 2008_보안서비스 미작동으로 인한 아이템 도난 배상 요구 【사건개...


### 8) 프롬프트 — 콘텐츠 분쟁 해결 전문가

In [9]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """당신은 디지털 콘텐츠(게임, 이러닝, 웹콘텐츠) 분쟁 해결을 돕는 전문 상담가입니다.
아래 [분쟁조정 사례] 문서 내용만을 근거로, 사용자의 상황에 맞는 해결 방향을 친절하고 구체적으로 안내하세요.

[분쟁조정 사례]
{context}

답변 형식:
1. 상황 분석: 사용자 분쟁의 유형(게임/이러닝/웹콘텐츠)과 핵심 쟁점
2. 관련 법령·약관: 사례에 언급된 법령이나 약관 조항 (예: 전자상거래법, 민법, 소비자분쟁해결기준)
3. 유사 사례: 비슷한 조정 사례의 연도, 기관, 결과를 간단히 소개
4. 해결 방안: 사용자가 취할 수 있는 단계별 조치 (사업자 요구 → 분쟁조정 신청 등)
5. 참고 사항: 결과가 달라질 수 있는 조건이나 추가로 확인할 점

답변 규칙:
- 문서에 있는 내용만 사용하고, 법령 조항이나 사례를 지어내지 마세요.
- 관련 사례가 문서에 전혀 없을 때에만 "제공된 사례집에서 관련 사례를 찾을 수 없습니다"라고 답하세요.
- 마지막에 "※ 본 답변은 과거 분쟁조정 사례에 근거한 참고 정보이며 법률 자문이 아닙니다."를 덧붙이세요."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

### 9) LCEL RAG 체인 — 검색 1회로 답변 + 출처 함께 반환
`RunnableParallel` 로 검색 결과(`docs`)를 한 번만 만든 뒤, 같은 문서로 답변을 생성하므로 답변의 근거와 출처가 일치합니다.

In [10]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs) -> str:
    """검색된 청크를 페이지 번호와 함께 하나의 문자열로 합침"""
    return "\n\n".join(f"[P{d.metadata.get('page_label', '?')}]\n{d.page_content}" for d in docs)


llm = get_llm(temperature=0.2)  # 문서 기반 답변: 낮은 temperature

answer_chain = (
    {"context": lambda x: format_docs(x["docs"]), "question": lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())  # 1) 검색 1회
    | RunnablePassthrough.assign(answer=answer_chain)                  # 2) 같은 docs 로 답변 생성
)
print("  LCEL RAG 파이프라인 구축 완료!")


def ask(question: str, show_sources: int = 3):
    """질문 → 답변과 참조 페이지 출력"""
    result = rag_chain.invoke(question)
    print(f"Q: {question}\n")
    print(f"A: {result['answer']}\n")
    print(f"  참조: {len(result['docs'])}개 청크")
    for j, doc in enumerate(result["docs"][:show_sources], 1):
        snippet = doc.page_content[:80].replace("\n", " ")
        print(f"    {j}. P{doc.metadata.get('page_label', '?')}: {snippet}...")
    return result

[LLM 생성] provider=openai | model=gpt-4o
  LCEL RAG 파이프라인 구축 완료!


### 10) 테스트 — 분야별 분쟁 상황 질문

In [11]:
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임 회사가 복구를 거부하고 있어요. 어떻게 해야 하나요?",
    "인터넷 동영상 강의를 결제했는데 중도에 해지하고 싶어요. 환불받을 수 있나요? 위약금은 얼마나 되나요?",
    "무료 이벤트에 참여했는데 이후 동의 없이 자동으로 소액 결제가 되었습니다. 환급받을 수 있나요?",
    "미성년자인 자녀가 부모 동의 없이 게임 아이템을 결제했어요. 취소할 수 있나요?",
    "불법 프로그램을 사용했다는 이유로 게임 계정이 영구 정지되었는데, 저는 사용한 적이 없어요.",
]

print("=" * 70)
print(" 콘텐츠분쟁해결 RAG 시스템 테스트")
print("=" * 70)
for i, q in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/{len(test_questions)}】")
    ask(q)
    print("-" * 70)

 콘텐츠분쟁해결 RAG 시스템 테스트

【테스트 1/5】
Q: 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임 회사가 복구를 거부하고 있어요. 어떻게 해야 하나요?

A: 1. 상황 분석: 사용자는 온라인 게임에서 시스템 오류로 인해 아이템이 소멸된 상황입니다. 핵심 쟁점은 시스템 오류로 인한 아이템 소멸에 대한 복구 가능 여부입니다.

2. 관련 법령·약관: 게임사의 약관에 따라 계정의 명의자 여부가 복구 처리에 영향을 미칠 수 있습니다. 또한, 게임사의 운영정책에 따라 시스템 오류로 인한 피해 보상 여부가 결정될 수 있습니다.

3. 유사 사례:
   - 2006년 사례(P9): 시스템 오류로 소멸된 아이템 복구 요구. 계정 명의자가 아닌 경우 복구가 불가하다는 답변을 받음.
   - 2006년 사례(P10): 프로그램 오류로 소멸된 아이템 복구 요구. 프로그램 오류가 인정되어 복구가 이루어짐.

4. 해결 방안:
   - 먼저, 게임사의 고객센터에 문의하여 시스템 오류로 인한 아이템 소멸에 대한 복구 요청을 다시 시도해보세요. 이때, 계정 명의자가 직접 요청하는 것이 중요합니다.
   - 만약 게임사가 복구를 거부할 경우, 한국소비자원이나 관련 기관에 분쟁조정을 신청하여 도움을 받을 수 있습니다. 이때, 시스템 오류로 인한 소멸 사실을 증명할 수 있는 자료를 준비하는 것이 좋습니다.

5. 참고 사항: 계정의 명의자가 아닌 경우 복구가 어려울 수 있으므로, 계정 명의자와 협력하여 복구 요청을 진행하는 것이 중요합니다. 또한, 게임사의 약관 및 운영정책을 확인하여 복구 가능 여부를 판단하는 것이 필요합니다.

※ 본 답변은 과거 분쟁조정 사례에 근거한 참고 정보이며 법률 자문이 아닙니다.

  참조: 5개 청크
    1. P9: 1. 게임 3게임1 ■ 1-가-1) 한국소비자원 조정전 상담사례 2006_시스템 오류로 소멸된 아이템 복구 요구 【사건개요 】 신청인은 피신청인...
    2. P31: 1. 게임 25■ 1-가-1) 한국소비자원 조정전

### 11) 분쟁 유형 분류 함수 (선택) — 키워드 기반
질문에 포함된 키워드로 분쟁 유형을 **게임 / 이러닝 / 웹콘텐츠 / 기타** 로 분류합니다.
위에서부터 순서대로 검사하므로, 여러 유형의 키워드가 함께 있으면 먼저 검사한 유형(게임)으로 분류됩니다.

In [12]:
def classify_dispute_type(query):
    """질문의 키워드로 분쟁 유형 분류"""
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]

    query_lower = query.lower()  # 영문 키워드가 섞여도 대소문자 구분 없이 비교

    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"


# 테스트 질문 분류 결과 확인 (LLM 호출 없음)
for q in test_questions + ["영화 예매권을 샀는데 사이트에서 등록이 거절됐어요."]:
    print(f"[{classify_dispute_type(q)}] {q}")

[게임] 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임 회사가 복구를 거부하고 있어요. 어떻게 해야 하나요?
[이러닝] 인터넷 동영상 강의를 결제했는데 중도에 해지하고 싶어요. 환불받을 수 있나요? 위약금은 얼마나 되나요?
[기타] 무료 이벤트에 참여했는데 이후 동의 없이 자동으로 소액 결제가 되었습니다. 환급받을 수 있나요?
[게임] 미성년자인 자녀가 부모 동의 없이 게임 아이템을 결제했어요. 취소할 수 있나요?
[게임] 불법 프로그램을 사용했다는 이유로 게임 계정이 영구 정지되었는데, 저는 사용한 적이 없어요.
[웹콘텐츠] 영화 예매권을 샀는데 사이트에서 등록이 거절됐어요.


#### 분류 결과를 RAG 에 활용
분류한 유형을 질문 앞에 붙여(`[게임 분쟁] ...`) 같은 분야의 사례가 검색되도록 돕습니다. `기타` 는 질문을 그대로 사용합니다.

In [13]:
def ask_with_type(question: str):
    """분쟁 유형을 분류한 뒤 유형 정보를 붙여 RAG 실행"""
    dispute_type = classify_dispute_type(question)
    print(f"▶ 분쟁 유형: {dispute_type}\n")
    query = question if dispute_type == "기타" else f"[{dispute_type} 분쟁] {question}"
    return ask(query)


ask_with_type("영화 예매권을 샀는데 사이트에서 등록이 거절됐어요.")

▶ 분쟁 유형: 웹콘텐츠

Q: [웹콘텐츠 분쟁] 영화 예매권을 샀는데 사이트에서 등록이 거절됐어요.

A: 1. 상황 분석: 사용자의 분쟁은 웹콘텐츠 유형에 해당하며, 핵심 쟁점은 영화 예매권의 등록 거절과 관련된 문제입니다. 예매권의 유효기간이 경과되었거나, 유효기간이 명시되지 않아 발생한 문제일 수 있습니다.

2. 관련 법령·약관: 소비자기본법에 따른 소비자분쟁해결기준이 관련될 수 있습니다. 이 기준은 예매권과 같은 유가증권의 유효기간 문제를 다루고 있습니다.

3. 유사 사례: 2007년 한국소비자원 조정결정례에서는 예매권의 유효기간이 명시되지 않았고, 유효기간이 경과된 경우에도 예매권을 사용할 수 있도록 조정된 사례가 있습니다. 피청구인은 청구인에게 예매권을 등록하여 사용할 수 있도록 조치하였습니다.

4. 해결 방안:
   - 먼저 예매권을 발행한 업체에 문의하여 유효기간 및 등록 거절 사유를 확인하세요.
   - 예매권에 유효기간이 명시되지 않았거나, 유효기간이 지나치게 짧았다면, 해당 업체에 예매권 등록을 요구할 수 있습니다.
   - 요구가 받아들여지지 않을 경우, 소비자분쟁조정위원회에 분쟁조정을 신청할 수 있습니다.

5. 참고 사항: 예매권의 유효기간이 명시되어 있었는지, 그리고 예매권을 발행한 업체의 정책에 따라 결과가 달라질 수 있습니다. 예매권을 받은 시점과 유효기간을 정확히 확인하는 것이 중요합니다.

※ 본 답변은 과거 분쟁조정 사례에 근거한 참고 정보이며 법률 자문이 아닙니다.

  참조: 5개 청크
    1. P99: 3. 기타(웹) 93기타(웹) 3 ■ 3-가 한국소비자원 조정결정례 2007_등록 거절된 영화예매권 이행 요구 【사건개요 】 청구인은 인터넷서비...
    2. P100: 【처리경위 】 이 사건 예매권은 청구인이 발행자인 피청구인에게 제시하여 증표에 기재된 내용에 따라 물 품 또는 용역을 제공받을 수 있는 유가증권...
    3. P56: 본건은 2008. 9. 22. 신청인의 미성년 아들이 신청인의 이름을

{'docs': [Document(id='1b3c325e-fe46-4bbd-b6b2-04777bf7c869', metadata={'producer': 'Hancom PDF 1.3.0.410', 'creator': 'Hancom PDF 1.3.0.410', 'creationdate': '2011-01-20T18:01:33+09:00', 'title': '제 2절 영국사례', 'moddate': '2011-01-20T18:01:33+09:00', 'pdfversion': '1.4', 'source': '../data/콘텐츠분쟁해결_사례.pdf', 'total_pages': 109, 'page': 98, 'page_label': '99'}, page_content='3. 기타(웹)\n93기타(웹) 3\n■ 3-가 한국소비자원 조정결정례\n2007_등록 거절된 영화예매권 이행 요구\n【사건개요 】\n청구인은 인터넷서비스업체인 ‘○○파워콤 ’에 회원으로 가입하면서 피청구인이 발행한 영화예매권\n(이하 ‘예매권“이라 함)을 사은품으로 받았는바 , 이를 사용하기 위해 2006. 6. 5. 피청구인 홈페이지에\n서 등록하려고 하였으나 등록이 거절되어 피청구인에게 확인해보니 예매권 유효기간 3개월이 경과되\n어 사용할 수 없다고 하였다 . 청구인은 예매권에 유효기간에 대한 구체적인 내용이 표시되어 있지 \n않아 피청구인이 정한 유효기간 내에 사용할 수 없었을 뿐 아니라 , 유효기간이 3개월로 너무 단기로 \n설정되어 예매권을 사용할 수 없었으므로 예매권 등록을 통한 이행을 요구하는 반면, 피청구인은 \n청구인이 보유한 예매권은 ‘○○파워콤 ’에 판매한 것을 ‘○○파워콤 ’에서 사은품으로 교부한 것이고 , \n유효기간은 당사와 ‘○○파워콤 간 약정에 의해 정해지는데 구매시점으로부터 최장 3개월로 규정되\n어 있으며 , 구매 기업별로 유효기간이 다르기 때문에 예매권에 일률적으로 유효기간을 표시할 수 \n없으므로 청구인의 요구를 수용할 수 없다고 하여 분쟁이 발생하게 되었다 .\n

#### (참고) LLM 기반 분류와 비교
키워드 방식은 빠르고 비용이 없지만, 키워드가 없는 문장은 `기타` 가 되고 "환불"처럼 여러 분야에 쓰이는 단어는 잘못 분류될 수 있습니다.
문제 2-2 의 `EnumOutputParser` 로 LLM 이 문맥을 보고 분류하게 하면 이런 경우를 보완할 수 있습니다.

In [14]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser  # langchain 1.x: classic 패키지에서 import


class DisputeType(Enum):
    GAME = "게임"
    ELEARNING = "이러닝"
    WEB = "웹콘텐츠"


type_parser = EnumOutputParser(enum=DisputeType)
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 콘텐츠 분쟁 질문을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("human", "{question}"),
]).partial(format_instructions=type_parser.get_format_instructions())

classify_chain = classify_prompt | get_llm(temperature=0, verbose=False) | type_parser

compare_questions = [
    "결제한 지 하루 만에 환불을 요청했는데 거절당했어요.",          # "환불" → 키워드 방식은 이러닝
    "영화 예매권을 샀는데 등록이 안 돼요.",                      # 키워드 없음 → 키워드 방식은 기타
    "부모님 몰래 아이템을 샀는데 취소할 수 있나요?",
]
for q in compare_questions:
    llm_type = classify_chain.invoke({"question": q}).value
    print(f"키워드: {classify_dispute_type(q):<5} | LLM: {llm_type:<5} | {q}")

키워드: 이러닝   | LLM: 웹콘텐츠  | 결제한 지 하루 만에 환불을 요청했는데 거절당했어요.
키워드: 기타    | LLM: 웹콘텐츠  | 영화 예매권을 샀는데 등록이 안 돼요.
키워드: 게임    | LLM: 게임    | 부모님 몰래 아이템을 샀는데 취소할 수 있나요?


### 12) 직접 질문하기

In [15]:
question = input("분쟁 상황을 입력하세요: ").strip() or "온라인 학습 서비스를 휴학 중에 해지하면 위약금을 내야 하나요?"
ask(question)

Q: 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?

A: 1. 상황 분석: 사용자는 온라인 게임에서 시스템 오류로 인해 아이템이 소실된 상황입니다. 핵심 쟁점은 시스템 오류로 인한 아이템 소실에 대한 복구 가능 여부입니다.

2. 관련 법령·약관: 게임사의 약관에 따라 시스템 오류로 인한 피해 보상 여부가 결정될 수 있습니다. 일반적으로 게임사의 약관에는 시스템 오류로 인한 피해에 대한 보상 규정이 포함되어 있을 수 있습니다.

3. 유사 사례:
   - [P9] 2006년 사례에서는 시스템 오류로 소멸된 아이템에 대해 사용자가 복구를 요구했으나, 계정 명의자가 아니어서 복구가 거부되었습니다.
   - [P29] 2009년 사례에서는 시스템 오류로 인한 게임머니 소실을 주장했으나, 유사한 불만이 접수되지 않아 시스템 오류로 보기 어렵다는 이유로 복구가 거부되었습니다.

4. 해결 방안:
   - 먼저, 게임사의 고객센터에 시스템 오류로 인한 아이템 소실에 대해 공식적으로 복구 요청을 하세요. 이때, 오류 발생 시점과 상황을 구체적으로 설명하는 것이 중요합니다.
   - 게임사의 약관을 확인하여 시스템 오류로 인한 피해 보상 규정이 있는지 살펴보세요.
   - 만약 게임사가 복구를 거부한다면, 한국소비자원이나 관련 소비자 보호 기관에 분쟁조정을 신청할 수 있습니다. 이때, 게임사의 약관과 본인의 상황을 상세히 설명하는 것이 도움이 될 수 있습니다.

5. 참고 사항: 
   - 계정 명의자 여부가 복구 가능 여부에 영향을 미칠 수 있습니다. 계정 명의자가 본인이 아닐 경우, 복구가 어려울 수 있습니다.
   - 시스템 오류가 아닌 다른 원인(예: 통신 문제)일 가능성도 있으므로, 본인의 인터넷 연결 상태나 PC 상태도 점검해보는 것이 좋습니다.

※ 본 답변은 과거 분쟁조정 사례에 근거한 참고 정보이며 법률 자문이 아닙니다.

  참조: 5개 청크
    1. P9: 1. 게임 3게임1 ■ 

{'docs': [Document(id='24f37323-b93c-46ee-aa6d-17ba60d5aa8f', metadata={'producer': 'Hancom PDF 1.3.0.410', 'creator': 'Hancom PDF 1.3.0.410', 'creationdate': '2011-01-20T18:01:33+09:00', 'title': '제 2절 영국사례', 'moddate': '2011-01-20T18:01:33+09:00', 'pdfversion': '1.4', 'source': '../data/콘텐츠분쟁해결_사례.pdf', 'total_pages': 109, 'page': 8, 'page_label': '9'}, page_content='1. 게임\n3게임1\n■ 1-가-1) 한국소비자원 조정전 상담사례\n2006_시스템 오류로 소멸된 아이템 복구 요구\n【사건개요 】\n신청인은 피신청인이 제공하는 온라인게임서비스를 이용하던 중 시스템 오류로 120,000 원 상당의 \n아이템이 소멸되었고 , 이에 피신청인에게 수차례 이의제기하였으나 사과 공지문만 안내 되고 아이템\n이 복구 되지 않았다 . 피신청인이 정상적으로 거래하던 중 아이템이 소멸된 것을 확인해 주었으나 \n규정상 복구가 불가하다는 답변을 한 것에 대하여 신청인은 정상적으로 이용하다 시스템 오류로 \n인해 소멸된 아이템에 대해 원상복구를 요구하였다 .\n【쟁점사항 】\n게임 중 시스템 오류로 인한 피해 보상 여부\n【처리경위 】\n피신청인은 신청인이 동 계정의 본소유자가 아니라며 처리를 거절하였다 .\n확인한 바, 신청인은 동 계정 명의가 타인으로 되어있으며 본인이 실소유주라고 주장하고 \n있으나 , 피신청인의 약관에 의하면 현금거래 및 계정공유를 인정하지 않도록 규정되어 있으\n므로, 이에 계정 명의자가 아닐 경우 복구 처리는 불가하다 .\n따라서 피신청인은 계정 명의자가 아이템의 복구 요청 시 처리가 가능하다며 계정 명의자가 \n직접 복구신청을 해줄 것을 요구하였다 .\n【처리결